# 📦 Optimisation des Stocks par Programmation Linéaire (MILP)

## Contexte et Problématique :
Ce projet s'inscrit dans une démarche d'amélioration de la performance supply chain . Actuellement, l'analyse des indicateurs de performance (KPIs) montre un écart significatif par rapport aux objectifs fixés (Target).
###  Le problème majeur identifié est le surstockage :
#### Coûts de possession élevés : Trop de capital est immobilisé dans les entrepôts.
#### Dépassement des Targets : Les niveaux de stock actuels sont bien au-dessus du stock de sécurité optimal.
#### Risques d'obsolescence : Une rotation de stock trop faible.
## Objectif du Modèle :     
L'objectif est de concevoir un outil d'aide à la décision capable de déterminer le plan d'approvisionnement optimal. Le modèle doit minimiser le coût total (Commande + Stockage) tout en garantissant la satisfaction de la demande.

En utilisant la Programmation Linéaire Mixte en Nombres Entiers (MILP), nous cherchons à répondre à deux questions critiques :

### Quand commander ? (Pour éviter les coûts fixes inutiles).
### Quelle quantité commander ? (Pour éviter le surstockage et atteindre les targets).

## Structure de la Solution :

Contrairement aux approches statiques, cet outil est dynamique et s'interface directement avec le système d'information de l'entreprise :

### Entrées : Importation automatique des données de demande et des paramètres de coûts via des fichiers Excel.
### Moteur d'optimisation : Modélisation mathématique avec Pyomo.
### Sorties : Calendrier d'approvisionnement optimal et visualisation des niveaux de stock futurs.


In [1]:
from pyomo.environ import *
import pandas as pd
import numpy as np


---
## 1. LES PARAMÈTRES



In [2]:
model = AbstractModel()

# ---------- Horizon temporel ----------
model.T   = Param(within=PositiveIntegers)          # dernière période (fin septembre)
model.PER = RangeSet(1, model.T)                    # périodes 1..T
model.t_juillet = Param(within=PositiveIntegers)    # dernier mois avant fermeture fournisseur
model.AoutSet    = Set(within=model.PER)             # période(s) de fermeture fournisseur

# ---------- Ensembles de références ----------
model.MP = Set()   # Matières Premières
model.SF = Set()   # Semi-Finis
model.PF = Set()   # Produits Finis

# ---------- Paramètres économiques généraux ----------
model.TARGET = Param()     # cible fiscale fin d'horizon ($)
model.tau     = Param()    # taux de possession annuel
model.z_alpha = Param()    # coefficient niveau de service
model.M       = Param()    # big-M (linéarisation MOQ/MLS)
model.V_max   = Param()    # capacité de stockage globale (m3)

# ---------- Paramètres par référence — Matières Premières ----------
model.prix_MP       = Param(model.MP)
model.stock_init_MP = Param(model.MP)
model.LT            = Param(model.MP)                 # lead time fournisseur
model.MOQ           = Param(model.MP)
model.volume_MP     = Param(model.MP)
model.SS_MP         = Param(model.MP)                  # stock de sécurité (calculé via EOQ, injecté en data)

# ---------- Paramètres par référence — Semi-Finis ----------
model.prix_SF       = Param(model.SF)
model.stock_init_SF = Param(model.SF)
model.ct_SF         = Param(model.SF)                  # cycle de production
model.MLS_SF        = Param(model.SF)
model.charge_SF     = Param(model.SF)
model.volume_SF     = Param(model.SF)
model.SS_SF         = Param(model.SF)

# ---------- Paramètres par référence — Produits Finis ----------
model.prix_PF       = Param(model.PF)
model.stock_init_PF = Param(model.PF)
model.ct_PF         = Param(model.PF)
model.MLS_PF        = Param(model.PF)
model.charge_PF     = Param(model.PF)
model.volume_PF     = Param(model.PF)
model.SS_PF         = Param(model.PF)

# ---------- Demande client (Produits Finis uniquement) ----------
model.demande = Param(model.PF, model.PER)

# ---------- Pipeline déjà engagé (figé, extrait SAP) ----------
model.OC     = Param(model.MP, model.PER)   # commandes MP déjà passées, en transit
model.WIP_SF = Param(model.SF, model.PER)   # encours production SF déjà lancé
model.WIP_PF = Param(model.PF, model.PER)   # encours production PF déjà lancé

# ---------- Nomenclature BOM ----------
model.a = Param(model.MP, model.SF, default=0)   # MP -> SF
model.b = Param(model.SF, model.PF, default=0)   # SF -> PF

# ---------- Capacités de production ----------
model.Cap_SF = Param(model.PER)
model.Cap_PF = Param(model.PER)


---
## 2. LES VARIABLES


In [3]:
# --- Stocks en fin de période ---
model.S_MP = Var(model.MP, model.PER, domain=NonNegativeReals)
model.S_SF = Var(model.SF, model.PER, domain=NonNegativeReals)
model.S_PF = Var(model.PF, model.PER, domain=NonNegativeReals)

# --- Quantités commandées (MP) / produites (SF, PF) ---
model.O_MP = Var(model.MP, model.PER, domain=NonNegativeReals)
model.P_SF = Var(model.SF, model.PER, domain=NonNegativeReals)
model.P_PF = Var(model.PF, model.PER, domain=NonNegativeReals)

# --- Variables binaires (déclenchement commande / OF) ---
model.x    = Var(model.MP, model.PER, domain=Binary)
model.y_SF = Var(model.SF, model.PER, domain=Binary)
model.y_PF = Var(model.PF, model.PER, domain=Binary)


---
## 3. LA FONCTION OBJECTIF (et les contraintes C1 → C13)

Toujours écrites uniquement avec les *noms* des paramètres — jamais leurs valeurs.


In [4]:
# ---------- Fonction objectif ----------
def cost_rule(m):
    return (
        sum(m.prix_MP[i]*m.S_MP[i,t] for i in m.MP for t in m.PER)
      + sum(m.prix_SF[j]*m.S_SF[j,t] for j in m.SF for t in m.PER)
      + sum(m.prix_PF[l]*m.S_PF[l,t] for l in m.PF for t in m.PER)
    )
model.cost = Objective(rule=cost_rule, sense=minimize)


In [5]:
# ---------- C1 — Bilan de stock Matière Première ----------
def c1_rule(m, i, t):
    S_prev = m.stock_init_MP[i] if t == 1 else m.S_MP[i, t-1]
    t_cmd  = t - m.LT[i]
    reception = m.O_MP[i, t_cmd] if t_cmd >= 1 else 0
    conso = sum(m.a[i,j]*m.P_SF[j,t] for j in m.SF if m.a[i,j] > 0)
    return m.S_MP[i,t] == S_prev + m.OC[i,t] + reception - conso
model.C1_Bilan_MP = Constraint(model.MP, model.PER, rule=c1_rule)

# ---------- C2 — Bilan de stock Semi-Fini ----------
def c2_rule(m, j, t):
    S_prev = m.stock_init_SF[j] if t == 1 else m.S_SF[j, t-1]
    t_lanc = t - m.ct_SF[j]
    reception = m.P_SF[j, t_lanc] if t_lanc >= 1 else 0
    conso = sum(m.b[j,l]*m.P_PF[l,t] for l in m.PF if m.b[j,l] > 0)
    return m.S_SF[j,t] == S_prev + m.WIP_SF[j,t] + reception - conso
model.C2_Bilan_SF = Constraint(model.SF, model.PER, rule=c2_rule)

# ---------- C3 — Bilan de stock Produit Fini ----------
def c3_rule(m, l, t):
    S_prev = m.stock_init_PF[l] if t == 1 else m.S_PF[l, t-1]
    t_lanc = t - m.ct_PF[l]
    reception = m.P_PF[l, t_lanc] if t_lanc >= 1 else 0
    return m.S_PF[l,t] == S_prev + m.WIP_PF[l,t] + reception - m.demande[l,t]
model.C3_Bilan_PF = Constraint(model.PF, model.PER, rule=c3_rule)

# C4 — stocks initiaux : déjà intégrés dans C1/C2/C3 via la condition t==1 (pas de contrainte séparée)

# ---------- C5 — Cible de valeur de stock fin d'année fiscale ----------
def c5_rule(m):
    return (
        sum(m.prix_MP[i]*m.S_MP[i, m.T] for i in m.MP)
      + sum(m.prix_SF[j]*m.S_SF[j, m.T] for j in m.SF)
      + sum(m.prix_PF[l]*m.S_PF[l, m.T] for l in m.PF)
      <= m.TARGET
    )
model.C5_Cible_Fiscale = Constraint(rule=c5_rule)

# ---------- C6 — Taux de service minimum (Produit Fini) ----------
def c6_rule(m, l, t):
    return m.S_PF[l,t] >= m.SS_PF[l]
model.C6_TauxService = Constraint(model.PF, model.PER, rule=c6_rule)

# ---------- C7 — Capacité de production ----------
def c7_sf_rule(m, t):
    return sum(m.charge_SF[j]*m.P_SF[j,t] for j in m.SF) <= m.Cap_SF[t]
model.C7_Capacite_SF = Constraint(model.PER, rule=c7_sf_rule)

def c7_pf_rule(m, t):
    return sum(m.charge_PF[l]*m.P_PF[l,t] for l in m.PF) <= m.Cap_PF[t]
model.C7_Capacite_PF = Constraint(model.PER, rule=c7_pf_rule)

# ---------- C8 — Capacité de stockage globale (volume) ----------
def c8_rule(m, t):
    return (
        sum(m.volume_MP[i]*m.S_MP[i,t] for i in m.MP)
      + sum(m.volume_SF[j]*m.S_SF[j,t] for j in m.SF)
      + sum(m.volume_PF[l]*m.S_PF[l,t] for l in m.PF)
      <= m.V_max
    )
model.C8_Capacite_Stockage = Constraint(model.PER, rule=c8_rule)

# ---------- C9 — MOQ et liaison binaire — Achat MP ----------
def c9a_rule(m, i, t):
    return m.O_MP[i,t] >= m.MOQ[i]*m.x[i,t]
model.C9a_MOQ_min = Constraint(model.MP, model.PER, rule=c9a_rule)

def c9b_rule(m, i, t):
    return m.O_MP[i,t] <= m.M*m.x[i,t]
model.C9b_MOQ_max = Constraint(model.MP, model.PER, rule=c9b_rule)

# ---------- C10 — MLS et liaison binaire — Production SF / PF ----------
def c10a_sf_rule(m, j, t):
    return m.P_SF[j,t] >= m.MLS_SF[j]*m.y_SF[j,t]
model.C10a_MLS_SF_min = Constraint(model.SF, model.PER, rule=c10a_sf_rule)

def c10b_sf_rule(m, j, t):
    return m.P_SF[j,t] <= m.M*m.y_SF[j,t]
model.C10b_MLS_SF_max = Constraint(model.SF, model.PER, rule=c10b_sf_rule)

def c10a_pf_rule(m, l, t):
    return m.P_PF[l,t] >= m.MLS_PF[l]*m.y_PF[l,t]
model.C10a_MLS_PF_min = Constraint(model.PF, model.PER, rule=c10a_pf_rule)

def c10b_pf_rule(m, l, t):
    return m.P_PF[l,t] <= m.M*m.y_PF[l,t]
model.C10b_MLS_PF_max = Constraint(model.PF, model.PER, rule=c10b_pf_rule)

# ---------- C11 — Fermeture fournisseur en août (aucune commande possible) ----------
def c11_rule(m, i, t):
    return m.O_MP[i,t] == 0
model.C11_Fermeture_Fournisseur = Constraint(model.MP, model.AoutSet, rule=c11_rule)

# ---------- C12 — Couverture de stock avant fermeture ----------
def c12_rule(m, i):
    conso_aout = sum(m.a[i,j]*m.P_SF[j,t] for t in m.AoutSet for j in m.SF if m.a[i,j] > 0)
    return m.S_MP[i, m.t_juillet] >= conso_aout
model.C12_Couverture_Aout = Constraint(model.MP, rule=c12_rule)

# C13 — non-négativité et domaine binaire : déjà intégrés via domain=NonNegativeReals / Binary


---
## 4. SOLVE



In [ ]:
def resoudre(instance, solveur="cbc", verbose=True):
    opt = SolverFactory(solveur)
    resultats = opt.solve(instance, tee=verbose)
    return resultats


---
## 5. METTRE EN PLACE LA DATA

Équivalent du bloc `data; ... end;` du `.mod` AMPL : **ce n'est qu'ici** qu'on injecte les
vraies valeurs, qu'on calcule les paramètres dérivés (EOQ / stock de sécurité), qu'on construit
l'instance concrète du modèle, qu'on résout, et qu'on affiche/trace les résultats.


### 5.1 Données fictives réalistes (MP / SF / PF / BOM / capacités)

In [ ]:
donnees_MP = {
    "MP1": {"prix": 50,  "stock_init": 5000,  "LT": 2, "MOQ": 500,  "sigma_D": 200, "C_passation": 300},
    "MP2": {"prix": 30,  "stock_init": 8000,  "LT": 1, "MOQ": 1000, "sigma_D": 150, "C_passation": 200},
    "MP3": {"prix": 120, "stock_init": 2000,  "LT": 2, "MOQ": 200,  "sigma_D": 80,  "C_passation": 500},
    "MP4": {"prix": 15,  "stock_init": 12000, "LT": 1, "MOQ": 2000, "sigma_D": 300, "C_passation": 100},
    "MP5": {"prix": 200, "stock_init": 1000,  "LT": 3, "MOQ": 100,  "sigma_D": 50,  "C_passation": 800},
}

donnees_SF = {
    "SF1": {"prix": 300, "stock_init": 800,  "ct": 1, "MLS": 100, "sigma_D": 60, "C_lancement": 400, "charge": 2.0},
    "SF2": {"prix": 180, "stock_init": 1200, "ct": 1, "MLS": 200, "sigma_D": 90, "C_lancement": 300, "charge": 1.5},
    "SF3": {"prix": 450, "stock_init": 400,  "ct": 2, "MLS": 50,  "sigma_D": 30, "C_lancement": 600, "charge": 3.0},
}

donnees_PF = {
    "PF1": {"prix": 800,  "stock_init": 300, "ct": 1, "MLS": 50,
            "demande": {1: 200, 2: 220, 3: 210, 4: 230},
            "sigma_D": 40, "C_lancement": 700, "charge": 4.0},
    "PF2": {"prix": 1200, "stock_init": 150, "ct": 1, "MLS": 30,
            "demande": {1: 100, 2: 110, 3: 105, 4: 120},
            "sigma_D": 25, "C_lancement": 900, "charge": 6.0},
}

a_bom = {  # MP -> SF
    "MP1": {"SF1": 2, "SF2": 1, "SF3": 0},
    "MP2": {"SF1": 0, "SF2": 3, "SF3": 1},
    "MP3": {"SF1": 1, "SF2": 0, "SF3": 2},
    "MP4": {"SF1": 3, "SF2": 2, "SF3": 0},
    "MP5": {"SF1": 0, "SF2": 0, "SF3": 1},
}
b_bom = {  # SF -> PF
    "SF1": {"PF1": 2, "PF2": 1},
    "SF2": {"PF1": 1, "PF2": 2},
    "SF3": {"PF1": 0, "PF2": 1},
}

OC = {
    "MP1": {1: 1000, 2: 500,  3: 0, 4: 0},
    "MP2": {1: 2000, 2: 1000, 3: 0, 4: 0},
    "MP3": {1: 300,  2: 200,  3: 0, 4: 0},
    "MP4": {1: 3000, 2: 1500, 3: 0, 4: 0},
    "MP5": {1: 100,  2: 0,    3: 0, 4: 0},
}
WIP_SF = {
    "SF1": {1: 200, 2: 0, 3: 0, 4: 0},
    "SF2": {1: 300, 2: 0, 3: 0, 4: 0},
    "SF3": {1: 50,  2: 0, 3: 0, 4: 0},
}
WIP_PF = {
    "PF1": {1: 100, 2: 0, 3: 0, 4: 0},
    "PF2": {1: 50,  2: 0, 3: 0, 4: 0},
}

volume_MP = {"MP1": 0.01, "MP2": 0.02, "MP3": 0.005, "MP4": 0.03, "MP5": 0.008}
volume_SF = {"SF1": 0.05, "SF2": 0.04, "SF3": 0.06}
volume_PF = {"PF1": 0.10, "PF2": 0.15}

Cap_SF = {1: 2000, 2: 2000, 3: 800, 4: 2000}   # août réduit (congés)
Cap_PF = {1: 1500, 2: 1500, 3: 600, 4: 1500}

noms_periodes = {1: "Juin", 2: "Juillet", 3: "Août", 4: "Septembre"}
print("✅ Données fictives chargées")


### 5.2 Calibration EOQ — calcul des stocks de sécurité `SS`

(pré-calcul, comme dans le notebook de modélisation — le résultat alimente `SS_MP` / `SS_SF` / `SS_PF`)


In [ ]:
tau, z_alpha = 0.20, 1.65

D_annuel_MP = {"MP1": 6000, "MP2": 12000, "MP3": 2400, "MP4": 20000, "MP5": 800}
D_annuel_SF = {"SF1": 2400, "SF2": 3600,  "SF3": 900}

SS_MP, SS_SF, SS_PF = {}, {}, {}
resultats_eoq = []

for i, p in donnees_MP.items():
    h = tau*p["prix"]
    Q_star = np.sqrt(2*D_annuel_MP[i]*p["C_passation"]/h)
    ss = z_alpha*p["sigma_D"]*np.sqrt(p["LT"])
    SS_MP[i] = ss
    resultats_eoq.append({"Référence": i, "Étage": "MP", "EOQ": round(Q_star), "SS": round(ss)})

for j, p in donnees_SF.items():
    h = tau*p["prix"]
    Q_star = np.sqrt(2*D_annuel_SF[j]*p["C_lancement"]/h)
    ss = z_alpha*p["sigma_D"]*np.sqrt(p["ct"])
    SS_SF[j] = ss
    resultats_eoq.append({"Référence": j, "Étage": "SF", "EOQ": round(Q_star), "SS": round(ss)})

for l, p in donnees_PF.items():
    h = tau*p["prix"]
    D_an = sum(p["demande"].values())*3
    Q_star = np.sqrt(2*D_an*p["C_lancement"]/h)
    ss = z_alpha*p["sigma_D"]*np.sqrt(p["ct"])
    SS_PF[l] = ss
    resultats_eoq.append({"Référence": l, "Étage": "PF", "EOQ": round(Q_star), "SS": round(ss)})

display(pd.DataFrame(resultats_eoq).set_index("Référence"))


### 5.3 Construction du `DataPortal` et de l'instance concrète

In [ ]:
data = DataPortal()

data['T'] = 4
data['t_juillet'] = 2
data['AoutSet'] = [3]

data['MP'] = list(donnees_MP.keys())
data['SF'] = list(donnees_SF.keys())
data['PF'] = list(donnees_PF.keys())

data['TARGET'] = 10_000_000
data['tau'] = tau
data['z_alpha'] = z_alpha
data['M'] = 1_000_000
data['V_max'] = 5000

data['prix_MP']       = {i: v["prix"] for i, v in donnees_MP.items()}
data['stock_init_MP'] = {i: v["stock_init"] for i, v in donnees_MP.items()}
data['LT']            = {i: v["LT"] for i, v in donnees_MP.items()}
data['MOQ']           = {i: v["MOQ"] for i, v in donnees_MP.items()}
data['volume_MP']     = volume_MP
data['SS_MP']         = SS_MP

data['prix_SF']       = {j: v["prix"] for j, v in donnees_SF.items()}
data['stock_init_SF'] = {j: v["stock_init"] for j, v in donnees_SF.items()}
data['ct_SF']         = {j: v["ct"] for j, v in donnees_SF.items()}
data['MLS_SF']        = {j: v["MLS"] for j, v in donnees_SF.items()}
data['charge_SF']     = {j: v["charge"] for j, v in donnees_SF.items()}
data['volume_SF']     = volume_SF
data['SS_SF']         = SS_SF

data['prix_PF']       = {l: v["prix"] for l, v in donnees_PF.items()}
data['stock_init_PF'] = {l: v["stock_init"] for l, v in donnees_PF.items()}
data['ct_PF']         = {l: v["ct"] for l, v in donnees_PF.items()}
data['MLS_PF']        = {l: v["MLS"] for l, v in donnees_PF.items()}
data['charge_PF']     = {l: v["charge"] for l, v in donnees_PF.items()}
data['volume_PF']     = volume_PF
data['SS_PF']         = SS_PF

data['demande'] = {(l, t): donnees_PF[l]["demande"][t] for l in donnees_PF for t in [1,2,3,4]}
data['OC']      = {(i, t): OC[i][t] for i in OC for t in [1,2,3,4]}
data['WIP_SF']  = {(j, t): WIP_SF[j][t] for j in WIP_SF for t in [1,2,3,4]}
data['WIP_PF']  = {(l, t): WIP_PF[l][t] for l in WIP_PF for t in [1,2,3,4]}

data['a'] = {(i, j): a_bom[i][j] for i in a_bom for j in a_bom[i] if a_bom[i][j] > 0}
data['b'] = {(j, l): b_bom[j][l] for j in b_bom for l in b_bom[j] if b_bom[j][l] > 0}

data['Cap_SF'] = Cap_SF
data['Cap_PF'] = Cap_PF

instance = model.create_instance(data)
print("✅ Instance concrète créée :", instance.name)


### 5.4 Résolution (appel réel de `resoudre`, défini à l'étape 4)

In [ ]:
resultats = resoudre(instance, solveur="cbc", verbose=False)

print("=" * 55)
print("  STATUT DE LA RÉSOLUTION")
print("=" * 55)
print(f"  Statut       : {resultats.solver.termination_condition}")
print(f"  Valeur objectif (Z) : {value(instance.cost):,.2f} $")
print("=" * 55)


### 5.5 Vérification de la cible fiscale (C5)

In [ ]:
T = instance.T
valeur_fin_sept = (
    sum(instance.prix_MP[i]*value(instance.S_MP[i, T]) for i in instance.MP)
  + sum(instance.prix_SF[j]*value(instance.S_SF[j, T]) for j in instance.SF)
  + sum(instance.prix_PF[l]*value(instance.S_PF[l, T]) for l in instance.PF)
)
print(f"Valeur de stock fin {noms_periodes[T]} : {valeur_fin_sept:,.0f} $")
print(f"Cible fiscale (TARGET)              : {value(instance.TARGET):,.0f} $")
print("✅ cible respectée" if valeur_fin_sept <= value(instance.TARGET) else "❌ cible dépassée")


### 5.6 Extraction des résultats en DataFrames

In [ ]:
def extraire(var, refs, label):
    lignes = [{"Référence": r, "Période": noms_periodes[t], "t": t, label: value(var[r,t])}
              for r in refs for t in instance.PER]
    df = pd.DataFrame(lignes)
    display(df.pivot(index="Référence", columns="Période", values=label)[
        [noms_periodes[t] for t in instance.PER]
    ])
    return df

print("Stocks MP :");  df_S_MP = extraire(instance.S_MP, instance.MP, "Stock")
print("Stocks SF :");  df_S_SF = extraire(instance.S_SF, instance.SF, "Stock")
print("Stocks PF :");  df_S_PF = extraire(instance.S_PF, instance.PF, "Stock")
print("Commandes O_MP :"); df_O_MP = extraire(instance.O_MP, instance.MP, "O_MP")
print("Production P_SF :"); df_P_SF = extraire(instance.P_SF, instance.SF, "P_SF")
print("Production P_PF :"); df_P_PF = extraire(instance.P_PF, instance.PF, "P_PF")


### 5.7 Valeur de stock par période et graphiques

In [ ]:
valeurs = []
for t in instance.PER:
    v_mp = sum(instance.prix_MP[i]*value(instance.S_MP[i,t]) for i in instance.MP)
    v_sf = sum(instance.prix_SF[j]*value(instance.S_SF[j,t]) for j in instance.SF)
    v_pf = sum(instance.prix_PF[l]*value(instance.S_PF[l,t]) for l in instance.PF)
    valeurs.append({"Période": noms_periodes[t], "MP": v_mp, "SF": v_sf, "PF": v_pf,
                     "Total": v_mp+v_sf+v_pf})
df_valeur = pd.DataFrame(valeurs).set_index("Période")
display(df_valeur)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))
for col in ["MP", "SF", "PF"]:
    ax.plot(df_valeur.index, df_valeur[col], marker="o", label=col)
ax.plot(df_valeur.index, df_valeur["Total"], marker="s", linewidth=2.5, color="black", label="Total")
ax.axhline(value(instance.TARGET), color="red", linestyle="--",
           label=f"Cible fiscale ({value(instance.TARGET):,.0f} $)")
ax.set_title("Évolution de la valeur de stock par étage")
ax.set_xlabel("Période"); ax.set_ylabel("Valeur du stock ($)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### 5.8 Résumé final

In [ ]:
print("=" * 60)
print("  RÉSUMÉ DE LA RÉSOLUTION")
print("=" * 60)
print(f"  Statut solveur          : {resultats.solver.termination_condition}")
print(f"  Valeur objectif Z       : {value(instance.cost):,.0f} $")
print(f"  Valeur stock fin {noms_periodes[T]:<9}: {valeur_fin_sept:,.0f} $")
print(f"  Cible fiscale           : {value(instance.TARGET):,.0f} $")
print(f"  Cible respectée         : {'✅ Oui' if valeur_fin_sept <= value(instance.TARGET) else '❌ Non'}")
print("=" * 60)
